### **Lap Weather Analysis — Vamsi**

##### **Wet Track Indicator Fields**

The primary dataset used in this notebook is **`lap_weather_data_2018_2025.csv`**, which merges FastF1 lap data with interpolated weather readings for each lap.

The following fields are relevant to identifying wet-track conditions:

| Field | Type | Source | Description |
|-------|------|--------|-------------|
| `Rainfall` | `bool` | FastF1 weather telemetry | `True` when rain was detected during the lap. **Primary wet-track indicator.** |
| `Humidity` | `float` | FastF1 weather telemetry | Relative humidity (%) — elevated values often accompany rainfall |
| `TrackTemp` | `float` | FastF1 weather telemetry | Track surface temperature (°C) — drops sharply in wet conditions |
| `Compound` | `str` (lap data) | FastF1 lap telemetry | Intermediate (`INTERMEDIATE`) or wet (`WET`) tyre compounds confirm wet running even when `Rainfall` is `False` (drying track) |

**`Rainfall`** is the canonical boolean flag used throughout this notebook to split laps into rainy vs. dry groups.  
`cleaned_lap_data_2018_2025.csv` does **not** contain `Rainfall` — use `lap_weather_data_2018_2025.csv` for any wet/dry analysis.

##### **Imports**

In [1]:
%pip install plotly
%pip install nbformat


import pandas as pd
import plotly.express as px
from scipy import stats

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


##### **Load Clean Laps**

- Filter lap data to only include laps without any track incidents, pit laps, or terminal laps
- Change `LapTime` and `SectorTime` to be seconds for later calculation and comparison
- Exclude columns not directly involved in Lap / Sector time analysis

In [2]:
time_cols = ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']
    
analysis_cols = [
    'Year', 'EventName', 'LapNumber', 'Driver', 'Team', 'Position', 
    'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 
    'Rainfall', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp',
    'WindSpeed', 'WindDirection'
]

# Load race data
race_data = pd.read_pickle('../data/f1_lap_weather_data.pkl')

# Convert time columns to seconds
for col in time_cols:
    race_data[col] = pd.to_timedelta(race_data[col]).dt.total_seconds()

clean_laps = (
    race_data.loc[
        (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
        & (race_data['IsTerminalLap'] == False),
        analysis_cols
    ]
)

print(f"Columns in dataset: {list(race_data.columns)}")
print(f"\nWet-track related fields present: {[c for c in ['Rainfall','Humidity','TrackTemp','Compound'] if c in race_data.columns]}")
print(f"\nRainfall value counts:\n{race_data['Rainfall'].value_counts()}")
clean_laps.head()

Columns in dataset: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'TrackStatus', 'Position', 'FastF1Generated', 'IsAccurate', 'Year', 'Location', 'EventName', 'LapStartTimeUTC', 'IsPitLap', 'IsTerminalLap', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 'WindSpeed', 'Rainfall']

Wet-track related fields present: ['Rainfall', 'Humidity', 'TrackTemp', 'Compound']

Rainfall value counts:
Rainfall
False    180236
True       8246
Name: count, dtype: int64


,Year,EventName,LapNumber,Driver,Team,Position,LapTime,Sector1Time,Sector2Time,Sector3Time,Rainfall,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,WindDirection
0,2018,Australian Grand Prix,1.0,GAS,Racing Bulls,17.0,105.060,41.610,25.495,37.955,False,24.200000,36.466667,997.083333,38.816667,2.350000,306.166667
1,2018,Australian Grand Prix,2.0,GAS,Racing Bulls,17.0,93.372,31.357,24.825,37.190,False,24.200000,36.300000,996.908197,38.257377,3.824590,296.901639
2,2018,Australian Grand Prix,3.0,GAS,Racing Bulls,17.0,92.861,31.160,24.725,36.976,False,24.003333,36.551667,997.100000,36.906667,3.990000,289.000000
3,2018,Australian Grand Prix,4.0,GAS,Racing Bulls,17.0,92.184,30.835,24.730,36.619,False,23.896667,36.280000,997.100000,36.793333,2.893333,253.500000
7,2018,Australian Grand Prix,8.0,GAS,Racing Bulls,16.0,91.319,30.438,24.594,36.287,False,23.826667,35.546667,997.000000,37.526667,3.220000,306.866667


##### **Adverse Weather Races**

Using the weather context gained from the Open-Meteo analysis and the existing weather features in FastF1 we can identify which races drivers faced adverse weather conditions. 

The Open-Meteo analysis highlights several races with adverse weather:
- **German Grand Prix 2019** - Most rain (5-hr total)
- **Azerbaijan Grand Prix - 2018** - Highest wind gust
- **Spanish Grand Prix - 2022** - Hottest race start

We can perform a similar analysis using the `Rainfall` boolean from FastF1 to understand the percentage of laps that drivers faced rainy conditions. These environmental factors are often associated with difficult driving conditions, limiting driver visibility and decreasing track grip. 

In [3]:
# Filter races that incurred rainfall
rainy_races = (
    clean_laps.copy()
    .groupby(['Year', 'EventName'], as_index=False)
    .agg(
        PercentRainfall=('Rainfall', lambda x: x.mean() * 100),
        TotalLaps=('Rainfall', 'count'),
        RainLaps=('Rainfall', 'sum')
    )
    .query('RainLaps > 0 and TotalLaps > 100')
    .sort_values('PercentRainfall', ascending=False)
    .reset_index(drop=True)
)

rainy_races.sort_values('PercentRainfall', ascending=False)

,Year,EventName,PercentRainfall,TotalLaps,RainLaps
0,2019,German Grand Prix,97.996661,599,587
1,2022,Japanese Grand Prix,87.338501,387,338
2,2019,Monaco Grand Prix,83.155300,1217,1012
3,2018,Spanish Grand Prix,82.965686,816,677
4,2024,São Paulo Grand Prix,67.770701,785,532
5,2024,British Grand Prix,39.176471,850,333
6,2025,Australian Grand Prix,33.147114,537,178
7,2024,Canadian Grand Prix,31.924360,899,287
8,2022,Monaco Grand Prix,29.383313,827,243
9,2021,Emilia Romagna Grand Prix,26.649418,773,206


In [4]:
# Reverse order for visual
rainy_races = rainy_races.sort_values('PercentRainfall', ascending=True)

# Add year + event name for clarity in visual
rainy_races['EventNameYear'] = rainy_races['Year'].astype(str) + ' ' + rainy_races['EventName'].str.replace('Grand Prix', 'GP')

# Create a bar chart for rainfall percentage
fig = px.bar(
    rainy_races,
    x='EventNameYear',
    y='PercentRainfall',
    orientation='v',
    hover_data=['RainLaps', 'TotalLaps'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={
        "text": "Event Name",
        "font": {"size": 15.5, "family": "Calibri", "color": "#000"}
    },
    yaxis_title={
        "text": "Percent of Laps with Rain",
        "font": {"size": 15.5, "family": "Calibri", "color": "#000"}
    },
    title={
        "text": "Top Rainiest Races (2018-2025)",
        "font": {"size": 18, "family": "Calibri Black", "color": "#000"}
    },
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100, r=100),
    width=1200
)

fig.update_xaxes(showline=True, linecolor="#000", linewidth=1.5)
fig.update_yaxes(showline=True, linecolor="#000", linewidth=1.5, gridcolor="#d9d9d9")
fig.show()

##### **Lap / Sector Time Comparison**

- `LapTimeDiff` represents the difference in a given driver's lap time and the median lap time of all drivers during that race
- `Sector(1-3)TimeDiff` represents the difference in a given driver's sector time and the median sector time of all drivers during that lap

In [5]:
# Calculate median lap time
median_laps = (
    clean_laps
    .groupby(['Year', 'EventName', 'LapNumber'], as_index=False)
    [time_cols]
    .agg('median')
    .rename({
        'LapTime': 'MedianLapTime',
        'Sector1Time': 'MedianSector1Time',
        'Sector2Time': 'MedianSector2Time',
        'Sector3Time': 'MedianSector3Time'
    }, axis=1)
)

# Merge clean laps with median
clean_laps = clean_laps.merge(median_laps, on=['Year', 'EventName', 'LapNumber'])

# Calculate time difference for each lap / sector
clean_laps['LapTimeDiff'] = clean_laps['LapTime'] - clean_laps['MedianLapTime']
clean_laps['Sector1TimeDiff'] = clean_laps['Sector1Time'] - clean_laps['MedianSector1Time']
clean_laps['Sector2TimeDiff'] = clean_laps['Sector2Time'] - clean_laps['MedianSector2Time']
clean_laps['Sector3TimeDiff'] = clean_laps['Sector3Time'] - clean_laps['MedianSector3Time']

clean_laps = clean_laps.drop(columns=['MedianLapTime', 'MedianSector1Time', 'MedianSector2Time', 'MedianSector3Time'])

clean_laps[['Year', 'EventName', 'Driver', 'LapNumber', 'LapTimeDiff', 'Sector1TimeDiff', 'Sector2TimeDiff', 'Sector3TimeDiff', 'Rainfall']].head()

,Year,EventName,Driver,LapNumber,LapTimeDiff,Sector1TimeDiff,Sector2TimeDiff,Sector3TimeDiff,Rainfall
0,2018,Australian Grand Prix,GAS,1.0,3.5320,2.903,0.6530,0.3660,False
1,2018,Australian Grand Prix,GAS,2.0,1.8070,0.771,0.2730,0.7000,False
2,2018,Australian Grand Prix,GAS,3.0,1.5755,0.573,0.3075,0.6245,False
3,2018,Australian Grand Prix,GAS,4.0,1.3320,0.475,0.3465,0.3080,False
4,2018,Australian Grand Prix,GAS,8.0,0.5600,0.170,0.3750,0.1510,False


##### **Performance Impact of Rainfall**

While it is generally understood that rainfall negatively impacts lap performance, we can still evaluate the strength and significance of this relationship using a t-test. We can also compare the mean performance differences observed during rainy and dry laps to better understand how adverse weather conditions affect drivers. By comparing these average differences, we can estimate the performance penalty associated with racing in rainy conditions.

- **Null Hypothesis:** There is no difference in driver performance between rainy and dry conditions.
- **Alternative Hypothesis:** Driver performance differs between rainy and dry conditions.

**Note:** One possible explanation for the non-significant Sector 1 results is the increased variability associated with race starts. Because Sector 1 experiences the highest levels of traffic and driver interaction, incidents such as spins, collisions, and defensive maneuvers may have a greater impact on sector times than weather conditions alone.

In [6]:
# Filter times for races with rain
rainy_race_times = clean_laps.merge(
    rainy_races[['Year', 'EventName']].drop_duplicates(),
    on=['Year', 'EventName'],
    how='inner'
)

rainy_race_results = []

for col in ['LapTimeDiff', 'Sector1TimeDiff', 'Sector2TimeDiff', 'Sector3TimeDiff']:
    rain_diffs = rainy_race_times.loc[
        rainy_race_times['Rainfall'] == True, col
    ].dropna()

    dry_diffs = rainy_race_times.loc[
        rainy_race_times['Rainfall'] == False, col
    ].dropna()
    
    t_stat, p_value = stats.ttest_ind(rain_diffs, dry_diffs, equal_var=False)
    
    rainy_race_results.append({
        'Metric': col,
        'AvgRainDiff': rain_diffs.mean(),
        'AvgDryDiff': dry_diffs.mean(),
        'RainPenalty': rain_diffs.mean() - dry_diffs.mean(),
        'TStat': t_stat,
        'PValue': p_value,
        'RainLaps': len(rain_diffs),
        'DryLaps': len(dry_diffs)
    })
    
rainy_race_results = pd.DataFrame(rainy_race_results)
rainy_race_results

,Metric,AvgRainDiff,AvgDryDiff,RainPenalty,TStat,PValue,RainLaps,DryLaps
0,LapTimeDiff,0.055763,-0.022049,0.077812,2.644139,0.008206,5505,12604
1,Sector1TimeDiff,0.024524,0.013620,0.010905,1.071172,0.284122,5505,12604
2,Sector2TimeDiff,0.042709,0.008356,0.034353,2.605827,0.009181,5505,12604
3,Sector3TimeDiff,0.034865,0.003207,0.031659,3.011972,0.002604,5505,12604


##### **Driver Performance in Adverse Weather Conditions**

Using the `LapTimeDiff` we can analyze the individual performance of drivers during adverse weather conditions. Drivers who are able to adapt quickly to changing conditions, manage tire grip effectively, and maintain consistency may gain a competitive advantage over the rest of the field.

- `RainPerformanceGain` - represents the average change in driver performance during rainy conditions relative to dry conditions. Positive values indicate the driver performed better relative to the field median in the rain, while negative values indicate worse performance.
- `MIN_RAIN_LAPS` - the minimum number of laps a driver must complete in the rain to be considered in the analysis

In [7]:
MIN_RAIN_LAPS = 75

# Driver performance in rainy laps
rain_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == True]
    .groupby('Driver')['LapTimeDiff']
    .agg(AvgRainDiff='mean', RainLapCount='count')
    .reset_index()
)

# Driver performance in dry laps
dry_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == False]
    .groupby('Driver')['LapTimeDiff']
    .agg(AvgDryDiff='mean', DryLapCount='count')
    .reset_index()
)

weather_performance = rain_performance.merge(dry_performance, on='Driver')

# Calculate performance gain
weather_performance['RainPerformanceGain'] = (
    weather_performance['AvgDryDiff']
    - weather_performance['AvgRainDiff']
)

# Filter out drivers with less than 75 rainy laps
weather_performance = (
    weather_performance
    .loc[weather_performance['RainLapCount'] > MIN_RAIN_LAPS]
).round(3)

weather_performance.head()

,Driver,AvgRainDiff,RainLapCount,AvgDryDiff,DryLapCount,RainPerformanceGain
0,ALB,0.429,182,0.189,5547,-0.240
1,ALO,0.046,219,0.064,5839,0.018
5,BOT,-0.137,296,-0.324,6554,-0.187
9,GAS,-0.154,236,0.184,7386,0.339
10,GIO,0.654,100,0.568,2677,-0.086


In [8]:
# Select top performers
top_10_rain_pace_drivers = weather_performance.sort_values('AvgRainDiff').head(10)
top_10_rain_pace_drivers = top_10_rain_pace_drivers.sort_values('AvgRainDiff', ascending=False)

fig = px.bar(
    top_10_rain_pace_drivers,
    x='Driver',
    y='AvgRainDiff',
    orientation='v',
    hover_data=['RainLapCount', 'DryLapCount'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={"text": "Driver", "font": {"size": 15.5, "family": "Calibri", "color": "#000"}},
    yaxis_title={"text": "Average Seconds", "font": {"size": 15.5, "family": "Calibri", "color": "#000"}},
    title={"text": "Top 10 Rainy Weather Drivers", "font": {"size": 18, "family": "Calibri Black", "color": "#000"}},
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100, r=50, b=60),
    width=600,
    height=400
)

fig.update_xaxes(showline=True, linecolor="#000", linewidth=1.5, side="top")
fig.update_yaxes(showline=True, linecolor="#000", linewidth=1.5, gridcolor="#d9d9d9")
fig.show()

In [9]:
rain_performance_gain = weather_performance.sort_values('RainPerformanceGain')

fig = px.bar(
    rain_performance_gain,
    x='Driver',
    y='RainPerformanceGain',
    orientation='v',
    hover_data=['RainLapCount', 'DryLapCount'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={"text": "Driver", "font": {"size": 15.5, "family": "Calibri", "color": "#000"}},
    yaxis_title={"text": "Performance Gain (sec)", "font": {"size": 15.5, "family": "Calibri", "color": "#000"}},
    title={"text": "Driver Performance Gain in the Rain", "font": {"size": 18, "family": "Calibri Black", "color": "#000"}},
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100),
    width=1000
)

fig.update_xaxes(showline=True, linecolor="#000", linewidth=1.5)
fig.update_yaxes(showline=True, linecolor="#000", linewidth=1.5, gridcolor="#d9d9d9")
fig.add_hline(y=0, line_color='black')
fig.show()

##### **Team Performance in Adverse Weather Conditions**

To evaluate team performance in adverse weather conditions, we aggregate `LapTimeDiff` values at the team level and compare performance during rainy and dry laps. Similar to the driver analysis, this approach measures how closely each team's drivers perform relative to the field median under different weather conditions.

**Possible Outcomes:**
- The impact of vehicle design on rainy-weather performance
- The effect of team strategies for handling changing track conditions on rainy-weather performance

In [10]:
MIN_TEAM_RAIN_LAPS = 150

# Team performance in rainy laps
team_rain_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == True]
    .groupby('Team')
    .agg(
        AvgRainDiff=('LapTimeDiff', 'mean'), 
        RainLapCount=('LapTimeDiff', 'count'), 
        DriverCountWet=('Driver', 'nunique')
    )
    .reset_index()
)

# Team performance in dry laps
team_dry_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == False]
    .groupby('Team')
    .agg(
        AvgDryDiff=('LapTimeDiff', 'mean'), 
        DryLapCount=('LapTimeDiff', 'count'), 
        DriverCountDry=('Driver', 'nunique')
    )
    .reset_index()
)

team_weather_performance = team_rain_performance.merge(team_dry_performance, on='Team')

team_weather_performance['RainPerformanceGain'] = (
    team_weather_performance['AvgDryDiff']
    - team_weather_performance['AvgRainDiff']
)

team_weather_performance = (
    team_weather_performance
    .loc[team_weather_performance['RainLapCount'] > MIN_TEAM_RAIN_LAPS]
).round(3)

team_weather_performance.sort_values('AvgRainDiff')

,Team,AvgRainDiff,RainLapCount,DriverCountWet,AvgDryDiff,DryLapCount,DriverCountDry,RainPerformanceGain
6,Mercedes,-1.044,609,4,-0.857,15866,4,0.187
8,Red Bull Racing,-0.836,617,6,-0.857,15214,7,-0.021
2,Ferrari,-0.584,490,5,-0.719,15193,6,-0.135
5,McLaren,-0.131,582,6,-0.260,15489,6,-0.128
0,Alpine,0.183,501,6,0.226,14851,8,0.043
7,Racing Bulls,0.269,553,9,0.328,14770,9,0.060
1,Aston Martin,0.309,530,5,0.162,15138,6,-0.148
3,Haas,0.540,551,7,0.646,14706,8,0.107
4,Kick Sauber,0.913,602,8,0.567,14894,9,-0.346
9,Williams,1.164,490,9,0.829,14597,11,-0.336


In [11]:
team_weather_performance.sort_values('RainPerformanceGain')

,Team,AvgRainDiff,RainLapCount,DriverCountWet,AvgDryDiff,DryLapCount,DriverCountDry,RainPerformanceGain
4,Kick Sauber,0.913,602,8,0.567,14894,9,-0.346
9,Williams,1.164,490,9,0.829,14597,11,-0.336
1,Aston Martin,0.309,530,5,0.162,15138,6,-0.148
2,Ferrari,-0.584,490,5,-0.719,15193,6,-0.135
5,McLaren,-0.131,582,6,-0.260,15489,6,-0.128
8,Red Bull Racing,-0.836,617,6,-0.857,15214,7,-0.021
0,Alpine,0.183,501,6,0.226,14851,8,0.043
7,Racing Bulls,0.269,553,9,0.328,14770,9,0.060
3,Haas,0.540,551,7,0.646,14706,8,0.107
6,Mercedes,-1.044,609,4,-0.857,15866,4,0.187
